In [14]:
import os
from ollama import chat


def list_markdown_files(directory: str) -> list[str]:
    """Возвращает список .md файлов в директории."""
    if not os.path.exists(directory):
        return []
    return [f for f in os.listdir(directory) if f.endswith(".md")]


def agent_query(question: str) -> str:
    # LLM получает список файлов и выбирает подходящий ===
    directory = "./docs"
    files = list_markdown_files(directory)

    if not files:
        return "Нет .md файлов в директории docs/"
    else:
        print(f"1) Файлы, найденные в системе: {files}")

    # Формируем промпт
    prompt = f"""
    Вопрос: {question}
    Доступные файлы: {files}

    Выбери ОДИН файл, который, скорее всего, описывает тему из вопроса.
    Никаких пояснений, только имя файла.
    """
    try:
        print(f"2) Запускаем модель 'mistral' на следующем промпте:\n{prompt}")
        response = chat(model="mistral", messages=[{"role": "user", "content": prompt}])
        print(f"3) Получаем ответ от модели: {response.message.content}")
        filename = response.message.content.strip().strip('"').strip("'")
        print(f"4) Отчищаем ответ: {filename}")
        print(f"5) Проверяем, что файл {filename} действительно существует в {files}")
        if filename in files:
            return filename
        else:
            return "Не удалось найти подходящий файл."

    except Exception as e:
        return f"Ошибка: {e}"

In [15]:
question = "Какой файл описывает RAG?"
answer = agent_query(question=question)
print(f"\nОтвет агента на вопрос {question}:\n{answer}")

1) Файлы, найденные в системе: ['rag.md', 'agents.md', 'fastapi.md']
2) Запускаем модель 'mistral' на следующем промпте:

    Вопрос: Какой файл описывает RAG?
    Доступные файлы: ['rag.md', 'agents.md', 'fastapi.md']

    Выбери ОДИН файл, который, скорее всего, описывает тему из вопроса.
    Никаких пояснений, только имя файла.
    
3) Получаем ответ от модели:  rag.md
4) Отчищаем ответ: rag.md
5) Проверяем, что файл rag.md действительно существует в ['rag.md', 'agents.md', 'fastapi.md']

Ответ агента на вопрос Какой файл описывает RAG?:
rag.md


### Примечание

* В продакшене используйте **валидацию ответа LLM** (урок 4.4),
* Ограничьте доступ инструментов: **никогда не давайте LLM прямой доступ к** ```os.system.```
* **Предотвращение бесконечных циклов**

В архитектуре ReAct агент работает в цикле ```Reasoning → Action → Observation. ``` Если агент «застрял» (например, инструмент постоянно возвращает ошибку), цикл может повторяться бесконечно.

**Решение:** параметр max_steps (или max_iterations), который ограничивает количество шагов выполнения:

```step = 0
while step < max_steps:
    action = llm.decide(...)
    result = tools.execute(action)
    if result.success:
        break
    step += 1
else:
    raise RuntimeError(f"Agent exceeded max_steps ({max_steps})")

---

In [17]:
import json
from typing import Any


class ReActAgent:
    def __init__(self, llm: Any, tools: dict[str, callable]):
        self.llm = llm
        self.tools = tools
        self.state: dict[str, Any] = {"history": [], "done": False, "error_count": 0}
    
    
    def reason(self) -> dict[str, Any]:
        prompt = f"""
        Состояние: {json.dumps(self.state, ensure_ascii=False)[:1000]}
        Инструменты: {list(self.tools.keys())}
        
        Что сделать следующим шагом? Верни строго валидный JSON:
        {{"action": "tool_name", "params": {{...}}, "thought": "почему"}}
        Или {{"action": "finish", "result": "итоговый ответ"}}
        """
        response = self.llm.chat(prompt)
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            return {"action": "finish", "result": f"Ошибка парсинга ответа LLM: {response}"}
    
    
    def act(self, decision: dict[str, Any]) -> dict[str, Any] | None:
        if decision.get("action") == "finish":
            self.state["done"] = True
            self.state["result"] = decision.get("result")
            return None
        
        tool_name = decision.get("action")
        params = decision.get("params", {})
        
        if tool_name not in self.tools:
            return {"action": tool_name, "error": f"Инструмент '{tool_name}' не найден"}
        
        try:
            result = self.tools[tool_name](**params)
            return {"action": tool_name, "result": result, "error": None}
        except Exception as e:
            return {"action": tool_name, "error": f"Ошибка выполнения: {e}"}
    
    
    def observe(self, action_result: dict[str, Any] | None) -> None:
        if action_result is None:
            return
        
        self.state["history"].append(action_result)
        
        if action_result.get("error"):
            self.state["error_count"] = self.state.get("error_count", 0) + 1
            if self.state["error_count"] >= 3:
                self.state["done"] = True
                self.state["result"] = "Превышен лимит ошибок"
        else:
            self.state["error_count"] = 0
    
    
    def run(self, task: str, max_steps: int = 5) -> str:
        self.state["task"] = task
        self.state["done"] = False
        self.state["error_count"] = 0
        
        for step in range(max_steps):
            if self.state["done"]:
                break
            
            decision = self.reason()
            action_result = self.act(decision)
            self.observe(action_result)
        
        return self.state.get("result", "Лимит шагов исчерпан без результата")


def check_database_metrics(service: str) -> dict[str, Any]:
    """
    Получает метрики базы данных для указанного сервиса.
    Идемпотентен: одинаковые параметры дают одинаковый результат.
    """
    # В реальности здесь будет запрос к БД или мониторингу
    mock_data = {
        "payment": {"latency_ms": 450, "connections": 95, "errors": 12},
        "auth": {"latency_ms": 120, "connections": 40, "errors": 1},
    }
    return mock_data.get(service, {"error": f"Сервис '{service}' не найден"})

### Когда цикл ломается: антипаттерны
#### Антипаттерн 1: Отсутствие фидбека на этапе Observe
Агент выполняет действие, но результат не влияет на состояние. Следующий этап Reason работает с теми же данными → зацикливание.

#### Антипаттерн 2: Слишком сложный этап Reason
Промпт пытается заставить агента спланировать всю цепочку действий сразу. Результат — игнорирование промежуточных результатов и жёсткий план, который ломается при первой ошибке.

#### Антипаттерн 3: Инструменты без валидации
Инструмент принимает любые параметры и падает с исключением. Агент не понимает, почему действие не выполнено, и повторяет его.

#### Антипаттерн 4: Отсутствие обработки ошибок LLM-ответа
LLM может вернуть невалидный JSON или текст вместо структуры. Без try/except при парсинге агент упадёт с исключением.

### Примечание

* Цикл Reason → Act → Observe не требует использования фреймворков. Его можно реализовать на чистом Python, как показано в примере выше.
* Не все задачи требуют полного цикла. Для простых сценариев достаточно одного прохода Reason → Act без итераций.
* Для отладки зацикливания агента добавляйте логирование каждого шага: какой инструмент вызван, с какими параметрами, какой результат получен.